# Intelligent Social Media Analysis System
### Pinterest Post Popularity Prediction and Topic Classification
> Full ML pipeline: data cleaning - feature engineering - Random Forest - YOLOv8 object detection - LSTM topic classifier - Flask API

## 1. Imports & Setup

In [1]:
import os
import re
import json
import numpy as np
import pandas as pd
from PIL import Image
from joblib import dump

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import LabelEncoder

print('All imports OK')

ModuleNotFoundError: No module named 'tensorflow'

## 2. Load & Filter Data

In [ ]:
pinterest = pd.read_csv('data/raw/pinterest.csv')
print('Columns:', pinterest.columns.tolist())
print('Shape:', pinterest.shape)
pinterest[['id', 'description', 'title', 'repin_count']].head()

In [ ]:
pinterest = pinterest.dropna(subset=['title', 'description', 'repin_count'])

pinterest['image_path'] = pinterest['id'].apply(
    lambda x: f'data/raw/images/images/{x}.jpg'
)
pinterest['has_image'] = pinterest['image_path'].apply(os.path.exists)
pinterest = pinterest[pinterest['has_image']].reset_index(drop=True)

os.makedirs('data/cleaned', exist_ok=True)
pinterest[['id', 'title', 'description', 'repin_count', 'image_path']].to_csv(
    'data/raw/pinterest_cleaned2.csv', index=False
)
print(f'Posts with valid images: {len(pinterest)}')

## 3. Text Cleaning & Feature Engineering

In [ ]:
pinterest = pd.read_csv('data/raw/pinterest_cleaned2.csv')
pinterest = pinterest.dropna(subset=['id', 'title', 'description', 'repin_count', 'image_path'])
pinterest = pinterest.drop_duplicates(subset=['title', 'description']).reset_index(drop=True)

pinterest['text'] = pinterest['title'].astype(str) + ' ' + pinterest['description'].astype(str)

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)           # remove URLs
    text = re.sub(r'[^a-zA-Z0-9# ]+', '', text)  # keep hashtags
    return text.strip()

pinterest['clean_text'] = pinterest['text'].apply(clean_text)
pinterest['hashtags'] = pinterest['clean_text'].apply(
    lambda x: [w for w in x.split() if w.startswith('#')]
)
pinterest['n_hashtags'] = pinterest['hashtags'].apply(len)
pinterest['text_len'] = pinterest['clean_text'].apply(len)

# Fix #1: expanded emoji range to cover all Unicode emoji blocks
EMOJI_PATTERN = re.compile(
    '[\U0001F300-\U0001F9FF\U00002600-\U000027BF]+', flags=re.UNICODE
)
pinterest['has_emoji'] = pinterest['clean_text'].apply(
    lambda x: int(bool(EMOJI_PATTERN.search(x)))
)

print(pinterest[['clean_text', 'n_hashtags', 'text_len', 'has_emoji']].head())

## 4. Image Preprocessing

In [ ]:
def process_image(path):
    # Fix #2: bare except replaced with specific exceptions
    try:
        with Image.open(path) as img:
            img = img.convert('RGB')
            img = img.resize((640, 640))
            img.save(path)
            return True
    except (OSError, IOError, ValueError) as e:
        print(f'Failed to process {path}: {e}')
        return False

pinterest['image_ok'] = pinterest['image_path'].apply(process_image)
pinterest = pinterest[pinterest['image_ok']].reset_index(drop=True)

pinterest_final = pinterest[['id', 'clean_text', 'repin_count', 'image_path',
                              'n_hashtags', 'text_len', 'has_emoji', 'hashtags']]
pinterest_final.to_csv('data/raw/pinterest_ready.csv', index=False)
print(f'Images processed successfully: {len(pinterest_final)}')

## 5. Labeling: Popularity & Topic

In [ ]:
# Popularity: binary label based on median repin count
median_repin = pinterest['repin_count'].median()
pinterest['is_popular'] = (pinterest['repin_count'] > median_repin).astype(int)
print(f'Popularity threshold (median): {median_repin:.0f} repins')
print(f'Popular: {pinterest["is_popular"].sum()} | Not popular: {(pinterest["is_popular"]==0).sum()}')

# Topic: keyword-based labeling
TOPIC_KEYWORDS = {
    'fashion':  ['dress', 'outfit', 'fashion', 'style', 'clothing', 'wear'],
    'food':     ['recipe', 'food', 'cake', 'kitchen', 'cook', 'meal', 'eat'],
    'art':      ['art', 'painting', 'drawing', 'illustration', 'design'],
    'home':     ['home', 'interior', 'furniture', 'decor', 'room', 'house'],
    'travel':   ['travel', 'beach', 'vacation', 'trip', 'explore', 'adventure'],
    'beauty':   ['beauty', 'makeup', 'skincare', 'cosmetics', 'hair'],
}

def extract_topic(text):
    text = text.lower()
    for topic, keywords in TOPIC_KEYWORDS.items():
        if any(kw in text for kw in keywords):
            return topic
    return 'other'

pinterest['topic'] = pinterest['clean_text'].apply(extract_topic)
pinterest.to_csv('data/raw/pinterest_labeled.csv', index=False)
print('\nTopic distribution:')
print(pinterest['topic'].value_counts())

## 6. Model v1 — Random Forest (Text Features Only)

In [ ]:
pinterest = pd.read_csv('data/raw/pinterest_labeled.csv')

features_v1 = ['text_len', 'n_hashtags', 'has_emoji', 'topic']
target = 'is_popular'

X = pinterest[features_v1]
y = pinterest[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = ['text_len', 'n_hashtags', 'has_emoji']
categorical_features = ['topic']

preprocessor = ColumnTransformer(transformers=[
    ('num', make_pipeline(SimpleImputer(strategy='mean'), StandardScaler()), numeric_features),
    ('cat', make_pipeline(OneHotEncoder(handle_unknown='ignore')), categorical_features),
])

model_v1 = make_pipeline(preprocessor, RandomForestClassifier(n_estimators=100, random_state=42))
model_v1.fit(X_train, y_train)

y_pred = model_v1.predict(X_test)
y_prob = model_v1.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f'ROC AUC: {roc_auc_score(y_test, y_prob):.4f}')

dump(model_v1, 'popularity_model.joblib')
print('Model v1 saved.')

## 7. YOLOv8 Object Detection — Enriching Features

In [ ]:
from ultralytics import YOLO

yolo_model = YOLO('yolov8n.pt')
pinterest = pd.read_csv('data/raw/pinterest_labeled.csv')

FOOD_KEYWORDS = {'banana', 'apple', 'pizza', 'cake', 'sandwich', 'hot dog',
                  'broccoli', 'carrot', 'orange', 'donut'}

def detect_objects(image_path):
    # Fix #2: specific exception instead of bare except
    try:
        results = yolo_model(image_path, verbose=False)
        if not results or len(results[0].boxes) == 0:
            return [], 0
        names = results[0].names
        detected = [names[int(cls)] for cls in results[0].boxes.cls]
        return detected, len(detected)
    except (FileNotFoundError, ValueError, RuntimeError) as e:
        print(f'Detection failed for {image_path}: {e}')
        return [], 0

object_lists, object_counts, has_person_col, has_food_col = [], [], [], []

for path in pinterest['image_path']:
    objs, count = detect_objects(path)
    object_lists.append(objs)
    object_counts.append(count)
    has_person_col.append(int('person' in objs))
    has_food_col.append(int(any(food in objs for food in FOOD_KEYWORDS)))

pinterest['object_list']  = object_lists
pinterest['object_count'] = object_counts
pinterest['has_person']   = has_person_col
pinterest['has_food']     = has_food_col

pinterest.to_csv('pinterest_with_objects.csv', index=False)
print('YOLO enrichment complete. Sample:')
print(pinterest[['image_path', 'object_count', 'has_person', 'has_food']].head())

## 8. Model v2 — Random Forest (Text + Image Features)

In [ ]:
pinterest = pd.read_csv('pinterest_with_objects.csv')

features_v2 = ['text_len', 'n_hashtags', 'has_emoji', 'topic',
                'has_person', 'has_food', 'object_count']
target = 'is_popular'

X = pinterest[features_v2]
y = pinterest[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features  = ['text_len', 'n_hashtags', 'has_emoji', 'object_count']
categorical_features = ['topic']
boolean_features  = ['has_person', 'has_food']

preprocessor = ColumnTransformer(transformers=[
    ('num', make_pipeline(SimpleImputer(strategy='mean'), StandardScaler()), numeric_features),
    ('cat', make_pipeline(OneHotEncoder(handle_unknown='ignore')), categorical_features),
    ('bool', 'passthrough', boolean_features),
])

model_v2 = make_pipeline(preprocessor, RandomForestClassifier(n_estimators=100, random_state=42))
model_v2.fit(X_train, y_train)

y_pred = model_v2.predict(X_test)
y_prob = model_v2.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f'ROC AUC: {roc_auc_score(y_test, y_prob):.4f}')

# Fix #3: consistent save path (was 'data/popularity_model_v2.joblib' but loaded as root-level in Flask)
dump(model_v2, 'popularity_model_v2.joblib')
print('Model v2 saved.')

## 9. LSTM — Topic Classification from Text

In [ ]:
pinterest = pd.read_csv('pinterest_with_objects.csv')
texts  = pinterest['clean_text'].astype(str).tolist()
labels = pinterest['topic'].astype(str).tolist()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)
y_cat = to_categorical(y)

MAX_NUM_WORDS      = 5000
MAX_SEQ_LENGTH     = 100
EMBEDDING_DIM      = 64
LSTM_UNITS         = 64

tokenizer = Tokenizer(num_words=MAX_NUM_WORDS)
tokenizer.fit_on_texts(texts)
X_seq = tokenizer.texts_to_sequences(texts)
X_pad = pad_sequences(X_seq, maxlen=MAX_SEQ_LENGTH)

X_train, X_test, y_train, y_test = train_test_split(
    X_pad, y_cat, test_size=0.2, random_state=42
)

# Fix #4: use Input layer explicitly (keras 3 deprecates input_length in Embedding)
model_lstm = Sequential([
    tf.keras.layers.Input(shape=(MAX_SEQ_LENGTH,)),
    Embedding(input_dim=MAX_NUM_WORDS, output_dim=EMBEDDING_DIM),
    LSTM(LSTM_UNITS),
    Dense(y_cat.shape[1], activation='softmax')
])

model_lstm.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_lstm.summary()

model_lstm.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10, batch_size=32,
    callbacks=[EarlyStopping(patience=2, restore_best_weights=True)],
    verbose=1
)

loss, accuracy = model_lstm.evaluate(X_test, y_test, verbose=0)
print(f'\nLSTM Test Accuracy: {accuracy:.4f}')

# Save model artifacts
model_lstm.save('topic_lstm_model.h5')

with open('topic_tokenizer.json', 'w') as f:
    f.write(tokenizer.to_json())

le_df = pd.DataFrame({'class': label_encoder.classes_,
                       'index': range(len(label_encoder.classes_))})
le_df.to_csv('topic_label_encoder.csv', index=False)

print('All LSTM artifacts saved.')

## 10. Flask API

Run as a standalone script (`app.py`). The code below is for reference — copy it to `app.py` and run with `python app.py`.

In [ ]:
# app.py — save this cell as a separate file

from flask import Flask, request, jsonify
import re
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
import json
from PIL import Image
from io import BytesIO
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import tokenizer_from_json
from ultralytics import YOLO

app = Flask(__name__)

# Load models once at startup
popularity_model = joblib.load('popularity_model_v2.joblib')
yolo_model       = YOLO('yolov8n.pt')
topic_model      = tf.keras.models.load_model('topic_lstm_model.h5')

# Fix #5: removed duplicate import inside 'with' block; load tokenizer correctly
with open('topic_tokenizer.json', 'r') as f:
    tokenizer = tokenizer_from_json(f.read())

le_df = pd.read_csv('topic_label_encoder.csv')
label_decoder = dict(zip(le_df['index'], le_df['class']))

MAX_SEQUENCE_LENGTH = 100
FOOD_OBJECTS = {'banana', 'apple', 'pizza', 'cake', 'sandwich', 'hot dog',
                 'broccoli', 'carrot', 'orange', 'donut'}


def extract_text_features(text):
    text_len    = len(text)
    n_hashtags  = text.count('#')
    has_emoji   = int(any(ord(c) > 10000 for c in text))
    return text_len, n_hashtags, has_emoji


def detect_image_objects(image):
    results      = yolo_model(image, verbose=False)[0]
    names        = results.names
    detected     = [names[int(cls)] for cls in results.boxes.cls]
    object_count = len(detected)
    has_person   = int('person' in detected)
    has_food     = int(any(obj in detected for obj in FOOD_OBJECTS))
    return detected, object_count, has_person, has_food


def predict_topic(text):
    seq  = tokenizer.texts_to_sequences([text])
    pad  = pad_sequences(seq, maxlen=MAX_SEQUENCE_LENGTH)
    pred = topic_model.predict(pad, verbose=0)
    return label_decoder[np.argmax(pred)]


@app.route('/analyze', methods=['POST'])
def analyze():
    text       = request.form.get('text', '').strip()
    image_file = request.files.get('image', None)

    # Fix #6: validate inputs with clear error messages
    if not text:
        return jsonify({'error': 'text field is required and cannot be empty.'}), 400
    if image_file is None:
        return jsonify({'error': 'image file is required.'}), 400

    try:
        topic                                          = predict_topic(text)
        text_len, n_hashtags, has_emoji                = extract_text_features(text)
        image                                          = Image.open(BytesIO(image_file.read())).convert('RGB')
        detected_objects, object_count, has_person, has_food = detect_image_objects(image)

        features = pd.DataFrame([{
            'text_len':     text_len,
            'n_hashtags':   n_hashtags,
            'has_emoji':    has_emoji,
            'topic':        topic,
            'has_person':   has_person,
            'has_food':     has_food,
            'object_count': object_count,
        }])

        popularity = int(popularity_model.predict(features)[0])
        prob       = float(popularity_model.predict_proba(features)[0][1])

        return jsonify({
            'topic':                  topic,
            'popularity_predicted':   popularity,
            'popularity_probability': round(prob, 3),
            'detected_objects':       detected_objects,
            'image_object_count':     object_count,
        })

    # Fix #6: catch prediction errors gracefully
    except Exception as e:
        return jsonify({'error': f'Prediction failed: {str(e)}'}), 500


if __name__ == '__main__':
    # Fix #7: disable debug=True for production
    app.run(debug=False, host='0.0.0.0', port=5000)